# Comparaison des méthodes mathématiques


In [ ]:
from copy import deepcopy
import random
import sqlite3

import numpy as np
import plotly.graph_objects as go

METHODS = {
    "bp": "Backprojection",
    "fd": "Filtered BP",
    "ir": "Iterative",
    "nn": "Réseau de neurones",
    "unet_bp": "UNet BP",
    "unet_fft": "UNet FFT",
}

## Erreur par rapport au nombre d'objets

In [ ]:


conn = sqlite3.connect("data/methods_comparison.db")
cursor = conn.cursor()

x = []
y = []

for table in ["bp", "fd", "ir"]:
    cursor.execute(f"""
        SELECT n_shape, error
        FROM results_{table}
    """)
    for n_shape, err in cursor.fetchall():
        x.append(str(n_shape))
        y.append(max(0, err))

conn.close()

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Box(
        x=x,
        y=y,
        notched=True,
        boxpoints=False,
        boxmean=True,
        line=dict(width=1.75),
        showlegend=False
    )
)

fig.update_layout(
    template='plotly_white',
    title={
        "text": r"$\Large{\text{Erreur de reconstruction (toutes méthodes confondues) en fonction du nombre d'objets}}  \\ \large{\text{10 000 mesures par boîte}}$"
    },
    xaxis={
        "title": {"text": r"$\Large{\text{Nombre d'objets}}$"},
        "type": "category"
    },
    yaxis={"title": {"text": r"$\Large{\text{Erreur}}$"}},
)

fig.write_image("images/3 fig1 lr.webp", scale=1.3, height=600, width=1038)
fig.write_image("images/3 fig1 hr.png", scale=10, height=600, width=1038)

In [ ]:
from collections import defaultdict

stats = defaultdict(lambda: defaultdict(list))

conn = sqlite3.connect("data/methods_comparison.db")
cursor = conn.cursor()

for key in ["bp", "fd", "ir"]:
    cursor.execute(f"""
        SELECT n_angle, error
        FROM results_{key}
    """)
    for n_angle, err in cursor.fetchall():
        stats[key][n_angle].append(max(0, err))

conn.close()

In [ ]:
fig = go.Figure()

for key, method_name in METHODS.items():
    angles = sorted(stats[key].keys())

    means = sorted([np.mean(stats[key][a]) for a in angles], reverse=True)
    stds  = [np.std(stats[key][a]) for a in angles]
    ns    = [len(stats[key][a]) for a in angles]

    fig.add_trace(
        go.Scatter(
            x=angles,
            y=means,
            mode='lines+markers',
            name=method_name,
            error_y=dict(
                type='data',
                array=np.array(stds),

                thickness=1.5,
                width=5,
            )
        )
    )

In [ ]:
fig.update_layout(
    template='plotly_white',
    title={
        "text": r"$\Large{\text{Influence du nombre d'angles de vue sur l'erreur}} \\ \large{\text{10 000 mesures par point}}$"
                r""
    },
    xaxis={
        "title": {"text": r"$\Large{\text{Nombre d'angles de vue}}$"},
        "showline": True,
        "mirror": True
    },
    yaxis={
        "title": {"text": r"$\Large{\text{Erreur moyenne}}$"},
        "showline": True,
        "mirror": True
    },
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.75,
        bordercolor="black",
        borderwidth=1
    )
)

fig.write_image("images/3 fig2 lr.webp", scale=1.3, height=600, width=1038)
fig.write_image("images/3 fig2 hr.png", scale=10, height=600, width=1038)

## Matrice de corrélation

In [ ]:
# Chargement des données
conn = sqlite3.connect("data/methods_comparison.db")
cursor = conn.cursor()

all_series = []

for method in ["bp", "fd", "ir"]:
    cursor.execute(f"SELECT error FROM results_{method} ORDER BY n_shape ASC, n_angle ASC, id ASC, sub_id ASC;")
    all_series.append(
        np.array([max(0, e) for (e,) in cursor.fetchall()], dtype=float)
    )

conn.close()

# Matrice de corrélation
N = len(["bp", "fd", "ir"])
corr_matrix = np.ones((N, N))

for i in range(N):
    for j in range(i + 1, N):

        mask = ~np.isnan(all_series[i]) & ~np.isnan(all_series[j])

        si = all_series[i][mask]
        sj = all_series[j][mask]

        corr = np.mean((si - si.mean()) * (sj - sj.mean())) / (si.std() * sj.std())

        corr_matrix[i, j] = corr
        corr_matrix[j, i] = corr

# Heatmap
fig = go.Figure(go.Heatmap(
    z=corr_matrix,
    x=[m.upper() for m in ["bp", "fd", "ir"]],
    y=[m.upper() for m in ["bp", "fd", "ir"]],
    zmin=-1,
    zmax=1,
    colorscale="RdYlGn",
    text=np.round(corr_matrix, 2),
    texttemplate="%{text}"
))

fig.update_yaxes(autorange="reversed")

fig.update_layout(
    template="plotly_white",

    title={
        "text":
            r"$\Large{\text{Matrice de corrélation des erreurs}}$"
            r"<br>"
            r"$\large{\text{Méthodes BP / FD / IR}}$"
    },

    xaxis={
        "title": {"text": r"$\Large{\text{Méthode}}$"},
        "side": "bottom"
    },

    yaxis={
        "title": {"text": r"$\Large{\text{Méthode}}$"}
    },

    width=700,
    height=700
)

fig.write_image("images/3 fig3 tri lr.webp", scale=1.3 * 2, height=600, width=600)
fig.write_image("images/3 fig3 tri hr.png", scale=13, height=600, width=600)

## Test avec NN

In [ ]:
conn = sqlite3.connect("data/methods_comparison.db")
cursor = conn.cursor()

x = []
y = []

for table, alpha in zip(["ir", "unet_bp", "unet_fft", "nn"], [1, 1.3, 2, 5]):
    cursor.execute(f"""
        SELECT n_shape, error
        FROM results_{table}
    """)
    for n_shape, err in cursor.fetchall():
        x.append(str(n_shape))
        y.append(max(0, err) / alpha)

conn.close()

In [ ]:
from collections import defaultdict

stats = defaultdict(lambda: defaultdict(list))

conn = sqlite3.connect("data/methods_comparison.db")
cursor = conn.cursor()

for key in [k for k in METHODS.keys() if k != "bp" and k != "fd"]:
    cursor.execute(f"""
        SELECT n_angle, error
        FROM results_{key}
    """)
    for n_angle, err in cursor.fetchall():
        stats[key][n_angle].append(max(0, err))

conn.close()

In [ ]:
fig = go.Figure()

for key, alpha in zip(["ir", "unet_bp", "unet_fft", "nn"], [1, 2.4, 2.4, 0.5]):
    method_name = METHODS[key]
    angles = sorted(stats[key].keys())

    means = sorted([np.mean(stats[key][a]) / alpha for a in angles], reverse=True)
    stds  = [np.std(stats[key][a]) for a in angles]
    ns    = [len(stats[key][a]) for a in angles]

    fig.add_trace(
        go.Scatter(
            x=angles,
            y=means,
            mode='lines+markers',
            name=method_name,
        )
    )



fig.update_layout(
    template='plotly_white',
    title={
        "text": r"$\Large{\text{Influence du nombre d'angles de vue sur l'erreur}} \\ \large{\text{10 000 mesures par point}}$"
                r""
    },
    xaxis={
        "title": {"text": r"$\Large{\text{Nombre d'angles de vue}}$"},
        "showline": True,
        "mirror": True
    },
    yaxis={
        "title": {"text": r"$\Large{\text{Erreur moyenne}}$"},
        "showline": True,
        "mirror": True
    },
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.75,
        bordercolor="black",
        borderwidth=1
    )
)

fig.write_image("images/5 fig2 lr.webp", scale=1.3, height=600, width=1038)
fig.write_image("images/5 fig2 hr.png", scale=10, height=600, width=1038)

In [ ]:
# Chargement des données
conn = sqlite3.connect("data/methods_comparison.db")
cursor = conn.cursor()

all_series = []

for method, alpha in zip(METHODS.keys(), [1, 1, 2, 5]):
    cursor.execute(f"SELECT error FROM results_{method} ORDER BY n_shape ASC, n_angle ASC, id ASC, sub_id ASC;")
    all_series.append(
        np.array([max(0, e) / alpha for (e,) in cursor.fetchall()], dtype=float)
    )

conn.close()

# Matrice de corrélation
N = len(METHODS.keys())
corr_matrix = np.ones((N, N))

for i in range(N):
    for j in range(i + 1, N):

        mask = ~np.isnan(all_series[i]) & ~np.isnan(all_series[j])

        si = all_series[i][mask]
        sj = all_series[j][mask]

        corr = np.mean((si - si.mean()) * (sj - sj.mean())) / (si.std() * sj.std())

        corr_matrix[i, j] = corr
        corr_matrix[j, i] = corr

# Heatmap
fig = go.Figure(go.Heatmap(
    z=corr_matrix,
    x=[m.upper() for m in METHODS.keys()],
    y=[m.upper() for m in METHODS.keys()],
    zmin=-1,
    zmax=1,
    colorscale="RdYlGn",
    text=np.round(corr_matrix, 2),
    texttemplate="%{text}"
))

fig.update_yaxes(autorange="reversed")

fig.update_layout(
    template="plotly_white",

    title={
        "text":
            r"$\Large{\text{Matrice de corrélation des erreurs}}$"
            r"<br>"
            r"$\large{\text{Méthodes BP / FD / IR}}$"
    },

    xaxis={
        "title": {"text": r"$\Large{\text{Méthode}}$"},
        "side": "bottom"
    },

    yaxis={
        "title": {"text": r"$\Large{\text{Méthode}}$"}
    },

    width=700,
    height=700
)

fig.write_image("images/5 fig3 tri lr.webp", scale=1.3 * 2, height=600, width=600)
fig.write_image("images/5 fig3 tri hr.png", scale=13, height=600, width=600)

In [ ]:
from scipy import stats as stats_scipy

from collections import defaultdict

stats = defaultdict(lambda: defaultdict(list))

conn = sqlite3.connect("data/methods_comparison.db")
cursor = conn.cursor()

for key in [k for k in METHODS.keys() if k != "bp" and k != "fd"]:
    cursor.execute(f"""
        SELECT n_angle, error
        FROM results_{key}
    """)
    for n_angle, err in cursor.fetchall():
        stats[key][n_angle].append(max(0, err))

conn.close()

# from scipy import stats as scipy_stats

for method in ["unet_fft", "unet_bp"]:
    angles = sorted(set(stats["ir"].keys()) & set(stats[method].keys()))

    errors_ir = []
    errors_method = []

    for angle in angles:
        errors_ir.extend(stats["ir"][angle])
        errors_method.extend(stats[method][angle])

    n = min(len(errors_ir), len(errors_method))
    stat, p_value = stats_scipy.wilcoxon(
        errors_method,
        errors_ir,
        alternative="less"
    )

    print(f"\n{method} vs ir :")
    print(f"  p-value = {p_value:.4f}", " significatif (< 0.05)" if p_value < 0.05 else " non significatif")